<a href="https://colab.research.google.com/github/sophiashiuu/DS340-Final-Project/blob/main/creating_log_mel_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def preprocess_audio(file_path):
    y, sr = librosa.load(file_path)

    # Standardize to 30 seconds
    target_length = 30 * sr

    if len(y) > target_length:
        y = y[:target_length]
    else:
        y = np.pad(y, (0, target_length - len(y)))

    # Generate mel spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr)

    # Convert to decibels
    log_mel = librosa.power_to_db(mel, ref=1.0)

    return log_mel

In [4]:
spectrograms = {}

folder_path = "/content/drive/MyDrive/DS 340/Final Project/PMEmo2019/chorus"

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)

    music_id = os.path.splitext(file_name)[0]

    spectrograms[music_id] = preprocess_audio(file_path)

print("Total spectrograms:", len(spectrograms))
print("Example shape:", next(iter(spectrograms.values())).shape)

Total spectrograms: 794
Example shape: (128, 1292)


In [5]:
spectrograms["1"]

array([[-44.615845 , -27.078783 , -15.620875 , ...,  20.33864  ,
         19.945368 ,  19.369078 ],
       [-44.615845 , -24.184296 , -13.475544 , ...,  21.938066 ,
         21.422962 ,  20.94615  ],
       [-44.615845 , -28.572147 , -22.441158 , ...,  11.07048  ,
         10.594926 ,  10.827566 ],
       ...,
       [-44.615845 , -44.615845 , -44.615845 , ..., -10.71209  ,
        -13.760279 , -15.5534315],
       [-44.615845 , -44.615845 , -44.615845 , ..., -12.285131 ,
        -17.651953 , -16.872812 ],
       [-44.615845 , -44.615845 , -44.615845 , ..., -22.166313 ,
        -30.876978 , -29.134018 ]], shape=(128, 1292), dtype=float32)

In [6]:
from google.colab import files

uploaded = files.upload()

Saving quadrant_data.csv to quadrant_data.csv


In [9]:
df = pd.read_csv("quadrant_data.csv")

In [10]:
df.head()

,musicId,Arousal(mean),Valence(mean),mood
0,1,0.4000,0.5750,1
1,4,0.2625,0.2875,0
2,5,0.1500,0.2000,0
3,6,0.5125,0.3500,2
4,7,0.7000,0.7250,3


In [20]:
X = []
y = []

for music_id, spectrogram in spectrograms.items():

  for index, row in df.iterrows():
    if int(music_id) == int(row["musicId"]):
      X.append(spectrogram)
      y.append(row["mood"])
      break

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("Labels shape:", y.shape)

X shape: (767, 128, 1292)
Labels shape: (767,)


In [21]:
save_path = "/content/drive/MyDrive/DS 340/Final Project"

np.save(os.path.join(save_path, "X.npy"), X)
np.save(os.path.join(save_path, "y.npy"), y)